# AI Functions: Customer Experience Telemetry

Turn raw chat threads, call transcripts, and support tickets into structured sentiment and topic telemetry — entirely in SQL with Snowflake AI Functions.

### What You'll Learn

- Land app UX telemetry (chat threads + thumbs up/down) via a stage → `VARIANT` → curated tables
- Score sentiment with `AI_SENTIMENT` (overall + per-aspect)
- Model topics with `AI_CLASSIFY`
- Extract structured fields with `AI_EXTRACT`
- Discover themes across thousands of rows with `AI_AGG` / `AI_SUMMARIZE_AGG`
- Surface churn-risk conversations with `AI_FILTER`
- Assemble a governed CX telemetry table and point AI Function Studio at a custom function

**Prerequisites:** Run `lab/setup.sql`, then `python lab/data_gen.py`, before starting this notebook.

---
## Section 1: Connect & Explore

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE ROLE ACCOUNTADMIN").collect()
session.sql("USE DATABASE FIELD_CX_DEMO").collect()
session.sql("USE SCHEMA AI_FUNCTIONS").collect()
session.sql("USE WAREHOUSE CX_AI_FUNCTIONS_WH").collect()
print("Connected to FIELD_CX_DEMO.AI_FUNCTIONS")

In [ ]:
# Peek at the raw conversations (loaded by data_gen.py)
session.sql("""
    SELECT thread_id, customer_id, channel, transcript
    FROM chat_threads
    LIMIT 5
""").show(max_width=120)

---
## Section 1b: App UX Telemetry — How Your App's Data Flows In

The `chat_threads` table above assumes clean telemetry already exists. In reality a conversational
app emits **message-level events** — every user turn, every assistant reply, and explicit
**feedback** (thumbs up / down). This section shows how that data flows from *your* app into
Snowflake, using two complementary patterns:

1. **Raw / semi-structured landing** — the app drops JSON files into a **stage**; `COPY INTO` loads
   them verbatim into a `VARIANT` column. Schema-on-read, captures everything, no schema coordination
   between app and warehouse. (The continuous production path is **Snowpipe** or **Snowpipe Streaming**.)
2. **Curated / structured** — `LATERAL FLATTEN` turns the raw payloads into typed, governed tables
   (`APP_THREADS`, `APP_MESSAGES`, `APP_FEEDBACK`) that join to `CUSTOMERS` and feed the AI functions
   and the semantic view.

Both coexist: **raw is the immutable landing zone, curated is what you serve.** We synthesize the
app's JSON below, but you can point your own app at the same stage and the rest of this flow is identical.


In [ ]:
# Simulate what your app emits: one JSON document per chat thread, with a messages[]
# array (user + assistant turns) and a feedback[] array (thumbs up / down). Then land
# those documents in the stage created by setup.sql — exactly where your app would drop them.

# 1) Message-level seed (topic + sentiment drive realistic text and feedback).
session.sql("""
    CREATE OR REPLACE TEMPORARY TABLE APP_EVENTS_SEED AS
    WITH base AS (
        SELECT
            'thr_' || LPAD(seq4() + 1, 5, '0')                              AS thread_id,
            UNIFORM(1, 500, RANDOM())                                       AS customer_id,
            ['web_chat','in_app','mobile'][UNIFORM(0, 2, RANDOM())]::STRING AS channel,
            '3.' || UNIFORM(1, 9, RANDOM())::STRING || '.0'                 AS app_version,
            DATEADD('minute', -UNIFORM(1, 260000, RANDOM()), CURRENT_TIMESTAMP()) AS started_at,
            UNIFORM(1, 100, RANDOM())                                       AS s,
            UNIFORM(0, 5, RANDOM())                                         AS topic_idx
        FROM TABLE(GENERATOR(ROWCOUNT => 150))
    ),
    labeled AS (
        SELECT base.*, CASE WHEN s <= 45 THEN 0 WHEN s <= 80 THEN 1 ELSE 2 END AS sentiment_idx
        FROM base
    )
    SELECT
        thread_id, customer_id, channel, app_version, started_at, sentiment_idx,
        GET(CASE topic_idx
            WHEN 0 THEN ARRAY_CONSTRUCT(
                'Your valuation is way off - it says my house is worth 80k less than three appraisers told me.',
                'The home value estimate was spot on, within a couple percent of my appraisal.',
                'How often does the estimated value refresh after I update the square footage?')
            WHEN 1 THEN ARRAY_CONSTRUCT(
                'I was charged twice this month and the Pro price jumped with no notice.',
                'Upgrading to Pro was worth it - the comps report alone pays for it.',
                'Can you explain the difference between the Starter and Pro plans?')
            WHEN 2 THEN ARRAY_CONSTRUCT(
                'I have been stuck on onboarding for an hour, the address import keeps failing.',
                'Setup was painless, I had a dashboard in ten minutes.',
                'Where is the guide for importing my B2B property portfolio?')
            WHEN 3 THEN ARRAY_CONSTRUCT(
                'The valuation chart is completely broken, it errors every time I open it.',
                'Thanks for the quick fix - the map view loads correctly now.',
                'Is the mobile app supposed to show the same comps as web?')
            WHEN 4 THEN ARRAY_CONSTRUCT(
                'I want to cancel immediately and get a refund, this is not working for me.',
                'I was going to cancel but the new market-trends feature convinced me to stay.',
                'If I cancel mid-cycle, do I keep access until the end of the period?')
            ELSE ARRAY_CONSTRUCT(
                'Every competitor has API access and you still do not. Dealbreaker for our team.',
                'Love the product - rental yield estimates would make it perfect.',
                'Are there plans to support commercial property valuations for B2B?')
        END, sentiment_idx)::STRING                                         AS user_text,
        GET(ARRAY_CONSTRUCT(
            'I hear you - I am escalating this to our team right now.',
            'Glad to hear it! Anything else I can help with?',
            'Here is what I found for you.'), sentiment_idx)::STRING        AS asst_text,
        'msg_' || thread_id || '_1'                                         AS user_msg_id,
        'msg_' || thread_id || '_2'                                         AS asst_msg_id,
        -- ~65% of threads leave feedback; negatives skew thumbs-down, positives thumbs-up
        CASE WHEN UNIFORM(1, 100, RANDOM()) <= 65
             THEN CASE sentiment_idx WHEN 0 THEN -1 WHEN 1 THEN 1
                       ELSE GET(ARRAY_CONSTRUCT(-1, 1), UNIFORM(0, 1, RANDOM()))::INT END
             ELSE NULL END                                                  AS rating
    FROM labeled
""").collect()

# 2) Serialize each thread to a single JSON document (messages[] + feedback[]).
session.sql("""
    CREATE OR REPLACE TEMPORARY TABLE APP_EVENTS_JSON AS
    SELECT OBJECT_CONSTRUCT(
        'thread_id',   thread_id,
        'customer_id', customer_id,
        'channel',     channel,
        'app_version', app_version,
        'started_at',  started_at::STRING,
        'messages', ARRAY_CONSTRUCT(
            OBJECT_CONSTRUCT('message_id', user_msg_id, 'turn_no', 1, 'role', 'user',
                             'content', user_text, 'created_at', started_at::STRING),
            OBJECT_CONSTRUCT('message_id', asst_msg_id, 'turn_no', 2, 'role', 'assistant',
                             'content', asst_text, 'model', 'llama3.1-8b',
                             'created_at', DATEADD('second', 20, started_at)::STRING)
        ),
        'feedback', IFF(rating IS NULL, ARRAY_CONSTRUCT(),
            ARRAY_CONSTRUCT(OBJECT_CONSTRUCT('message_id', asst_msg_id, 'rating', rating,
                             'created_at', DATEADD('second', 30, started_at)::STRING)))
    ) AS payload
    FROM APP_EVENTS_SEED
""").collect()

# 3) Drop the JSON docs into the stage (this is the step your app would do).
session.sql("""
    COPY INTO @APP_EVENTS_STAGE/app_events
    FROM (SELECT payload FROM APP_EVENTS_JSON)
    FILE_FORMAT = (TYPE = JSON)
    OVERWRITE = TRUE
""").collect()

session.sql("LIST @APP_EVENTS_STAGE/app_events").show(max_width=120)


### Load the raw events, as-is

The files in the stage are exactly what your app dropped. `COPY INTO` a `VARIANT` column preserves
the full payload — nothing is lost and the schema can evolve without breaking ingestion. In production
this manual `COPY` becomes **Snowpipe** (auto-ingest when a file arrives) or **Snowpipe Streaming**
(row-level, sub-second).


In [ ]:
# Load every JSON document into the raw VARIANT landing table, verbatim.
session.sql("TRUNCATE TABLE IF EXISTS RAW_APP_EVENTS").collect()
session.sql("""
    COPY INTO RAW_APP_EVENTS (payload, source_file)
    FROM (SELECT $1, METADATA$FILENAME FROM @APP_EVENTS_STAGE/app_events)
    FILE_FORMAT = (TYPE = JSON)
    ON_ERROR = 'ABORT_STATEMENT'
""").collect()

# One row per thread; the whole app payload is preserved in a single VARIANT column.
session.sql("SELECT source_file, payload FROM RAW_APP_EVENTS LIMIT 2").show(max_width=160)


### Curate into structured tables

`LATERAL FLATTEN` unpacks the nested `messages` and `feedback` arrays into typed rows. Keep the raw
table as your durable landing zone and use these curated tables for analytics, joins, and AI functions.

| | Raw `VARIANT` landing | Curated structured tables |
|---|---|---|
| **Shape** | One JSON doc per thread (`RAW_APP_EVENTS`) | `APP_THREADS` / `APP_MESSAGES` / `APP_FEEDBACK` |
| **Schema** | Schema-on-read (flexible) | Typed, governed, joinable to `CUSTOMERS` |
| **Best for** | Capturing everything the app sends | Analytics, AI functions, the semantic view |
| **Load path** | `COPY INTO` / Snowpipe / Snowpipe Streaming | `INSERT ... SELECT ... FLATTEN` (view, Task, or Dynamic Table) |


In [ ]:
# Flatten the raw payloads into the three curated tables defined in setup.sql.
session.sql("""
    INSERT OVERWRITE INTO APP_THREADS
    SELECT
        payload:thread_id::STRING,
        payload:customer_id::NUMBER,
        payload:channel::STRING,
        payload:app_version::STRING,
        payload:started_at::TIMESTAMP_NTZ
    FROM RAW_APP_EVENTS
""").collect()

session.sql("""
    INSERT OVERWRITE INTO APP_MESSAGES
    SELECT
        m.value:message_id::STRING,
        r.payload:thread_id::STRING,
        m.value:turn_no::NUMBER,
        m.value:role::STRING,
        m.value:content::STRING,
        m.value:model::STRING,
        m.value:created_at::TIMESTAMP_NTZ
    FROM RAW_APP_EVENTS r, LATERAL FLATTEN(input => r.payload:messages) m
""").collect()

session.sql("""
    INSERT OVERWRITE INTO APP_FEEDBACK
    SELECT
        f.value:message_id::STRING,
        r.payload:thread_id::STRING,
        f.value:rating::NUMBER,
        f.value:comment::STRING,
        f.value:created_at::TIMESTAMP_NTZ
    FROM RAW_APP_EVENTS r, LATERAL FLATTEN(input => r.payload:feedback) f
""").collect()

session.sql("""
    SELECT
        (SELECT COUNT(*) FROM APP_THREADS)  AS threads,
        (SELECT COUNT(*) FROM APP_MESSAGES) AS messages,
        (SELECT COUNT(*) FROM APP_FEEDBACK) AS feedback_events
""").show()


In [ ]:
# Close the loop: run an AI function on the curated app messages and correlate it with the
# explicit feedback. Does user sentiment line up with thumbs-down? (A real product-quality signal.)
session.sql("""
    SELECT
        AI_SENTIMENT(u.content):categories[0]:sentiment::STRING AS user_sentiment,
        COUNT(*)                                                AS user_messages,
        SUM(IFF(fb.rating < 0, 1, 0))                           AS thumbs_down,
        SUM(IFF(fb.rating > 0, 1, 0))                           AS thumbs_up
    FROM APP_MESSAGES u
    LEFT JOIN APP_FEEDBACK fb ON fb.thread_id = u.thread_id
    WHERE u.role = 'user'
    GROUP BY 1
    ORDER BY thumbs_down DESC
""").show()


### Feed the feedback into the governed metrics

Roll the thumbs up / down up to the customer level so the semantic view can expose a governed
`thumbs_down_rate`. After this runs, the **CX Intelligence agent** in the extensions notebook can
answer *"what's our thumbs-down rate by plan?"* off the same governed definition.


In [ ]:
# Rebuild the customer-level feedback rollup the semantic view reads (ANALYTICS.CUSTOMER_FEEDBACK).
session.sql("""
    CREATE OR REPLACE TABLE FIELD_CX_DEMO.ANALYTICS.CUSTOMER_FEEDBACK AS
    SELECT
        c.customer_id,
        COALESCE(f.thumbs_up, 0)   AS thumbs_up,
        COALESCE(f.thumbs_down, 0) AS thumbs_down
    FROM FIELD_CX_DEMO.ANALYTICS.CUSTOMERS c
    LEFT JOIN (
        SELECT t.customer_id,
               SUM(IFF(fb.rating > 0, 1, 0)) AS thumbs_up,
               SUM(IFF(fb.rating < 0, 1, 0)) AS thumbs_down
        FROM FIELD_CX_DEMO.AI_FUNCTIONS.APP_FEEDBACK fb
        JOIN FIELD_CX_DEMO.AI_FUNCTIONS.APP_THREADS  t ON fb.thread_id = t.thread_id
        GROUP BY t.customer_id
    ) f ON c.customer_id = f.customer_id
""").collect()
print("Rebuilt ANALYTICS.CUSTOMER_FEEDBACK — thumbs_down_rate in the semantic view now reflects live app telemetry.")


### From demo to production

- **Continuous ingestion:** swap the manual `COPY INTO` for **Snowpipe** (auto-ingest on file arrival)
  or **Snowpipe Streaming** (row-level, sub-second) so app events land as they happen.
- **Keep curation fresh:** wrap the `FLATTEN` inserts in a **Dynamic Table** or a scheduled **Task** so
  `APP_MESSAGES` / `APP_FEEDBACK` refresh automatically as new raw events arrive.
- **This is a template:** point your own app at a stage (or a streaming channel) with this JSON shape
  and the rest of the pipeline — AI functions, semantic view, agent — works unchanged.

---


---
## Section 2: Sentiment with AI_SENTIMENT

`AI_SENTIMENT(text, categories)` returns overall sentiment plus per-aspect sentiment in one call.

In [ ]:
# Overall + aspect sentiment for a sample of threads
session.sql("""
    SELECT
        thread_id,
        AI_SENTIMENT(transcript,
            ['valuation_accuracy','pricing_billing','onboarding']) AS sentiment
    FROM chat_threads
    LIMIT 5
""").show(max_width=140)

In [ ]:
# Materialize overall sentiment as a column for trending
session.sql("""
    CREATE OR REPLACE TABLE thread_sentiment AS
    SELECT
        thread_id,
        customer_id,
        created_at,
        AI_SENTIMENT(transcript):categories[0]:sentiment::string AS overall_sentiment
    FROM chat_threads
""").collect()

session.sql("""
    SELECT overall_sentiment, COUNT(*) AS threads
    FROM thread_sentiment
    GROUP BY overall_sentiment
    ORDER BY threads DESC
""").show()

---
## Section 3: Topic Modeling with AI_CLASSIFY

Map each conversation to a support taxonomy. Use `{'output_mode': 'multi'}` for threads that span topics.

In [ ]:
session.sql("""
    CREATE OR REPLACE TABLE classified_threads AS
    SELECT
        thread_id,
        customer_id,
        created_at,
        transcript,
        AI_CLASSIFY(transcript,
            ['valuation_accuracy','pricing_billing','onboarding',
             'bug_report','cancellation','feature_request'],
            {'task_description': 'Support topics for a home-valuation product'}
        ):labels[0]::string AS topic
    FROM chat_threads
""").collect()

session.sql("""
    SELECT topic, COUNT(*) AS threads
    FROM classified_threads
    GROUP BY topic
    ORDER BY threads DESC
""").show()

---
## Section 4: Structured Fields with AI_EXTRACT

Pull named fields out of free-text support tickets. Ask one clear question per field.

In [ ]:
session.sql("""
    SELECT
        ticket_id,
        AI_EXTRACT(
            text => body,
            responseFormat => {
                'intent': 'What does the customer want to do?',
                'product_area': 'Which product area is mentioned?',
                'urgency': 'Is this urgent? yes or no'
            }
        ):response AS fields
    FROM support_tickets
    LIMIT 5
""").show(max_width=140)

---
## Section 5: Theme Discovery with AI_AGG & AI_SUMMARIZE_AGG

The "unsupervised" view: surface dominant themes across thousands of rows. Both support `GROUP BY`.

In [ ]:
# Most common complaints per topic across the whole corpus
session.sql("""
    SELECT
        topic,
        AI_AGG(transcript,
            'Identify the most common customer complaints in these conversations') AS complaints
    FROM classified_threads
    GROUP BY topic
""").show(max_width=160)

In [ ]:
# A single executive summary of all support tickets
session.sql("""
    SELECT AI_SUMMARIZE_AGG(body) AS ticket_summary
    FROM support_tickets
""").show(max_width=200)

---
## Section 6: At-Risk Detection with AI_FILTER

Keep only churn-risk / escalation conversations, then join to structured data to prioritize by customer.

In [ ]:
session.sql("""
    SELECT
        t.thread_id,
        c.customer_id,
        c.customer_type,
        c.plan,
        c.region
    FROM chat_threads t
    JOIN customers c ON c.customer_id = t.customer_id
    WHERE AI_FILTER('This customer is frustrated and at risk of cancelling: ' || t.transcript)
    LIMIT 20
""").show()

---
## Section 7: Assemble the CX Telemetry Table

Combine sentiment + topic into one governed table you can trend in BI and feed to a churn model.

In [ ]:
session.sql("""
    CREATE OR REPLACE TABLE cx_telemetry AS
    SELECT
        s.thread_id,
        s.customer_id,
        s.created_at,
        s.overall_sentiment,
        c.topic
    FROM thread_sentiment s
    JOIN classified_threads c ON c.thread_id = s.thread_id
""").collect()

# Sentiment mix by topic — the telemetry Product/CX teams have been missing
session.sql("""
    SELECT topic, overall_sentiment, COUNT(*) AS threads
    FROM cx_telemetry
    GROUP BY topic, overall_sentiment
    ORDER BY topic, threads DESC
""").show(30)

---
## Section 8: AI Function Studio — create → evaluate → optimize

When a built-in isn't enough, you build a **custom AI function** and tune it in **AI Function Studio**
(Snowsight → **AI & ML → Cortex AI Function Studio**). Studio is backed by three stored procedures,
so you can run the exact same loop from SQL here and it shows up in the Studio UI:

1. **`CREATE_AI_FUNCTION`** — register a custom function on `AI_COMPLETE` (here: an escalation
   *router* that labels each conversation `LOW` / `MEDIUM` / `HIGH`).
2. **`EVALUATE_AI_FUNCTION`** — score it against a labeled test set with a metric (`exact_match`).
3. **`OPTIMIZE_AI_FUNCTION`** — auto-tune the prompt and compare models on an accuracy-vs-cost
   Pareto frontier, so you can pick the cheapest model that hits your quality bar.

> To demo the **UI** instead of SQL: open Cortex AI Function Studio in Snowsight, pick
> `ROUTE_ESCALATION` (created below), and run Evaluate / Optimize from the visual workflow.

In [ ]:
# Step 1 — CREATE: register a custom escalation router as a Studio function.
# This is the SQL behind the Studio "Create" UI; the function then appears in
# Snowsight -> AI & ML -> Cortex AI Function Studio.
session.sql("DROP FUNCTION IF EXISTS ROUTE_ESCALATION(VARCHAR)").collect()
session.sql("""
    CALL SNOWFLAKE.CORTEX.CREATE_AI_FUNCTION(
        'FIELD_CX_DEMO.AI_FUNCTIONS.ROUTE_ESCALATION',
        'llama3.1-8b',                                   -- baseline model (optimize compares others)
        $$You are a customer support triage assistant for a home-valuation (proptech) company. Classify the escalation priority of a customer conversation as exactly one of: LOW, MEDIUM, HIGH.
- HIGH: explicit cancellation or churn intent, legal/regulatory threats, or severe anger.
- MEDIUM: an unresolved problem or moderate frustration that needs human follow-up.
- LOW: informational, positive, or already-resolved conversations.
Respond with ONLY the single uppercase word: LOW, MEDIUM, or HIGH.$$,
        $$Conversation:
{TRANSCRIPT}$$,
        PARSE_JSON('[{"name":"TRANSCRIPT","sql_type":"VARCHAR"}]'),
        PARSE_JSON('[{"name":"priority","json_type":"string","description":"escalation priority: LOW, MEDIUM, or HIGH"}]'),
        'Route a customer conversation to an escalation priority tier (LOW/MEDIUM/HIGH)',
        NULL, NULL
    )
""").collect()

# Quick test on real threads
session.sql("""
    SELECT thread_id, ROUTE_ESCALATION(transcript) AS priority
    FROM chat_threads LIMIT 5
""").show()

### Step 2 — EVALUATE against a labeled test set

Studio needs a small **labeled** set (~20–50 rows): the input plus the *correct* answer. The cell below
builds a 24-row set of escalation transcripts hand-labeled `LOW` / `MEDIUM` / `HIGH`, then scores the
function with the `exact_match` metric. The returned `score` is your baseline accuracy.

In [ ]:
# Build a small hand-labeled test set (input TRANSCRIPT + gold EXPECTED label).
session.sql("""
    CREATE OR REPLACE TABLE ESCALATION_EVAL (TRANSCRIPT VARCHAR, EXPECTED VARCHAR) AS
    SELECT * FROM VALUES
    ('Hi, I just wanted to say the new valuation dashboard looks great. Thanks for the update!', 'LOW'),
    ('Customer: What time zone are your support hours in? Agent: 9-5 ET. Customer: Perfect, thanks.', 'LOW'),
    ('Just confirming my address change went through. Agent: Yes it is updated. Customer: Great.', 'LOW'),
    ('The report downloaded fine after I cleared my cache. All good now, thanks for the tip.', 'LOW'),
    ('Customer: Is there a mobile app? Agent: Yes, iOS and Android. Customer: Awesome, downloading now.', 'LOW'),
    ('Thanks for walking me through the comps feature, that answered my question completely.', 'LOW'),
    ('Quick one - can I export to CSV? Agent: Yes, top-right menu. Customer: Found it, thanks!', 'LOW'),
    ('No issues today, just checking my subscription renews in March. Agent: Correct. Customer: Thanks.', 'LOW'),
    ('My valuation looks off by about 10% versus a recent appraisal. Can someone take a look?', 'MEDIUM'),
    ('I have tried resetting my password twice and the email never arrives. Can you help?', 'MEDIUM'),
    ('The dashboard has been loading slowly for two days. It is usable but frustrating.', 'MEDIUM'),
    ('I was charged the annual plan but I thought I selected monthly. Can you check my billing?', 'MEDIUM'),
    ('The comps for my property include homes from a different neighborhood. That seems wrong.', 'MEDIUM'),
    ('I submitted a data correction last week and it still has not been applied. Any update?', 'MEDIUM'),
    ('Agent: Your report is generating. Customer: It has been 20 minutes and still nothing, annoying.', 'MEDIUM'),
    ('I need the API docs for the valuation endpoint - the link in your email is broken.', 'MEDIUM'),
    ('This is the third time my valuation is wrong. If it is not fixed today I am cancelling.', 'HIGH'),
    ('I have been overcharged three months in a row. I want a full refund or I dispute with my bank.', 'HIGH'),
    ('Your tool gave a valuation my lender rejected and I lost the deal. This is unacceptable.', 'HIGH'),
    ('I am done. Cancel my subscription immediately and delete my data. I will leave a review.', 'HIGH'),
    ('If you keep sharing my property data without consent I will contact my attorney.', 'HIGH'),
    ('Absolutely furious - your outage cost my brokerage a closing. I want a manager now.', 'HIGH'),
    ('I have emailed five times with no response. Fix this today or I cancel all our seats.', 'HIGH'),
    ('You billed my card after I cancelled. This is fraud and I am reporting you.', 'HIGH')
    AS t(TRANSCRIPT, EXPECTED)
""").collect()
session.sql("SELECT EXPECTED, COUNT(*) AS n FROM ESCALATION_EVAL GROUP BY EXPECTED ORDER BY EXPECTED").show()

In [ ]:
# EVALUATE the function (exact_match). Returns a VARIANT with the aggregate score.
# 12 positional params: function, test_table, input_cols, label_col, metric, model,
#                       sample_size, experiment, metric_options, max_length, custom_metric, run_id
session.sql("""
    CALL SNOWFLAKE.CORTEX.EVALUATE_AI_FUNCTION(
        'FIELD_CX_DEMO.AI_FUNCTIONS.ROUTE_ESCALATION',
        'FIELD_CX_DEMO.AI_FUNCTIONS.ESCALATION_EVAL',
        ARRAY_CONSTRUCT('TRANSCRIPT'), 'EXPECTED', 'exact_match', 'llama3.1-8b',
        NULL, NULL, NULL, 500, NULL, NULL
    )
""").collect()

# Pull the score + experiment name out of the CALL result
session.sql("""
    SELECT "EVALUATE_AI_FUNCTION":score::FLOAT          AS baseline_score,
           "EVALUATE_AI_FUNCTION":experiment_name::STRING AS experiment_name
    FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
""").show()

### Step 3 — OPTIMIZE: tune the prompt and compare models

`OPTIMIZE_AI_FUNCTION` auto-tunes the function body and runs the candidate **models concurrently**,
so you see an accuracy-vs-cost comparison and can pick the best trade-off. Pass **all** models in one
call — do not loop.

> **Runtime:** the optimizer typically runs **10+ minutes**. Run this cell live during the demo (or
> ahead of time). With `AUTO_BUDGET = 'demo'` it stays lightweight. Results are stored in the
> experiment and read back with `SHOW RUN METRICS` (last cell) — no need to re-run to inspect a model.

In [ ]:
# OPTIMIZE — 18 positional params. Runs ~10+ min; comparing 3 models on the Pareto frontier.
# 1 function, 2 training_table, 3 label, 4 input_cols, 5 metric, 6 models[], 7 reflection_model,
# 8 test_table, 9 budget, 10 validation_fraction, 11 temperature, 12 max_tokens, 13 metric_options,
# 14 custom_metric_udf, 15 run_id, 16 aggregation_metric, 17 optimize_mode, 18 experiment_name
session.sql("""
    CALL SNOWFLAKE.CORTEX.OPTIMIZE_AI_FUNCTION(
        'FIELD_CX_DEMO.AI_FUNCTIONS.ROUTE_ESCALATION',
        'FIELD_CX_DEMO.AI_FUNCTIONS.ESCALATION_EVAL',
        'EXPECTED',
        ARRAY_CONSTRUCT('TRANSCRIPT'),
        'exact_match',
        ARRAY_CONSTRUCT('llama3.1-8b', 'mistral-large2', 'claude-sonnet-4-6'),
        'claude-sonnet-4-6',
        NULL, 'demo', 0.5, 0.0, 8192, NULL, NULL, NULL, NULL, 'body',
        'ai_func_opt_ROUTE_ESCALATION_demo'
    )
""").collect()

# Read the per-model accuracy/cost results back from the experiment
session.sql("""
    SHOW RUN METRICS IN EXPERIMENT ai_func_opt_ROUTE_ESCALATION_demo
""").show()

In [ ]:
# Consolidated accuracy vs. cost comparison
# Join optimizer accuracy scores with actual credit usage per model

# Re-run SHOW RUN METRICS so RESULT_SCAN picks it up
session.sql("SHOW RUN METRICS IN EXPERIMENT ai_func_opt_ROUTE_ESCALATION_demo").collect()

accuracy_df = session.sql("""
    SELECT
        CASE SPLIT_PART("run_name", '_BEST', 1)
            WHEN 'LLAMA3_1_8B'      THEN 'llama3.1-8b'
            WHEN 'MISTRAL_LARGE2'   THEN 'mistral-large2'
            WHEN 'CLAUDE_SONNET_4_6' THEN 'claude-sonnet-4-6'
        END AS model_name,
        "value" AS valset_score
    FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
    WHERE "run_name" LIKE '%BEST'
""")

cost_df = session.sql("""
    SELECT 
        model_name,
        SUM(credits) AS total_credits,
        SUM(f.value:value::NUMBER) AS total_tokens
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY,
        LATERAL FLATTEN(input => metrics) f
    WHERE start_time >= DATEADD('hour', -2, CURRENT_TIMESTAMP())
      AND model_name IN ('llama3.1-8b', 'mistral-large2', 'claude-sonnet-4-6')
      AND function_name = 'AI_COMPLETE'
      AND f.value:key:unit = 'tokens'
    GROUP BY model_name
""")

consolidated = accuracy_df.join(cost_df, on='model_name', how='left')
consolidated.select(
    'model_name',
    'valset_score',
    'total_credits',
    'total_tokens'
).order_by('total_credits').show()

---
## Section 9: Cost, Usage & Guardrails

AI Functions bill by **tokens processed** (input + output), or by pages for document functions — **not** by warehouse size. A row-wise function like `AI_SENTIMENT` over a 1M-row table makes ~1M model calls, so the fastest way to burn credits is to run one across a full table without a `WHERE` or `LIMIT`.

This section shows how to **see** that spend and how to **cap** it:

- Monitor credits by function, model, and user via `SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY` (2–5 min latency).
- Enforce **per-user quotas** that auto-block AI requests at a credit limit.
- **Kill** a long-running / runaway query on demand.

> The `ACCOUNT_USAGE` view and the quota/cancel commands need elevated privileges (access to the `SNOWFLAKE` database, and `ACCOUNTADMIN` or a delegated role). If a cell errors on privileges in your demo account, treat it as reference SQL you'd run as an admin.

### Estimate token cost *before* a bulk run

Because AI Functions bill by **tokens processed**, you can price a full-table run *before* you launch it. Snowflake gives you two token-counting primitives:

| Primitive | Counts | Use when |
|---|---|---|
| `AI_COUNT_TOKENS('<function>', <input>)` | Tokens a **specific task function** will process, *including its internal prompt template* | Pricing one built-in (e.g. `AI_SENTIMENT`) — most accurate for that function |
| `SNOWFLAKE.CORTEX.COUNT_TOKENS('<model>', <text>)` | Raw input-text tokens for a **model** | A generic, model-based estimate that works for any function/model |

Both cost nothing to call and never invoke the model. The estimate → dollars chain is:

1. **Tokens** — `SUM(...)` across the rows you plan to process.
2. **Credits** — tokens × the model's *credits-per-million-tokens* rate (from the [Snowflake Service Consumption Table](https://www.snowflake.com/legal-files/CreditConsumptionTable.pdf)).
3. **Dollars** — credits × your negotiated *$ / credit*.

We'll do it two ways: a **rate-table projection** (a-priori), then **derive the rate from your own actuals** in `CORTEX_FUNCTIONS_QUERY_USAGE_HISTORY` and reconcile.

In [ ]:
# STEP 1 - Project total input tokens for a full-table AI_SENTIMENT run over CHAT_THREADS.
# AI_COUNT_TOKENS costs nothing to call and does not invoke the model - it just tokenizes.
token_estimate = session.sql("""
    SELECT
        COUNT(*)                                          AS rows_to_process,
        SUM(AI_COUNT_TOKENS('ai_sentiment', transcript))  AS total_input_tokens,
        AVG(AI_COUNT_TOKENS('ai_sentiment', transcript))::INT AS avg_tokens_per_row,
        MAX(AI_COUNT_TOKENS('ai_sentiment', transcript))  AS max_tokens_per_row
    FROM CHAT_THREADS
""")
token_estimate.show()

# Grab the projected token total into Python for the cost math below.
_est = token_estimate.collect()[0]
TOTAL_INPUT_TOKENS = int(_est['TOTAL_INPUT_TOKENS'])
ROWS_TO_PROCESS    = int(_est['ROWS_TO_PROCESS'])
print(f"\nFull run: {ROWS_TO_PROCESS:,} rows -> {TOTAL_INPUT_TOKENS:,} input tokens")
print("Note: AI_COUNT_TOKENS counts INPUT tokens (incl. the function's prompt template).")
print("For classify/sentiment the OUTPUT is a few tokens/row, so input dominates the bill.")

In [ ]:
# STEP 2 - A-priori projection using a model rate table.
# EDIT these to the current values from the Snowflake Service Consumption Table for your
# region/edition. They are illustrative placeholders, not a price quote.
CREDITS_PER_1M_TOKENS = {
    'llama3.1-8b':      0.19,   # small - cheapest, great for bounded classification
    'llama3.1-70b':     1.21,
    'claude-sonnet-4':  2.55,   # frontier - highest quality, highest $/token
}
PRICE_PER_CREDIT = 3.00        # <-- your negotiated USD per credit

import pandas as pd
rows = []
for model, rate in CREDITS_PER_1M_TOKENS.items():
    credits = TOTAL_INPUT_TOKENS / 1_000_000 * rate
    rows.append({
        'model': model,
        'credits_per_1M_tokens': rate,
        'projected_credits': round(credits, 4),
        'projected_usd': round(credits * PRICE_PER_CREDIT, 2),
    })
projection = pd.DataFrame(rows).sort_values('projected_usd')
print(f"Projected cost of AI_SENTIMENT over {ROWS_TO_PROCESS:,} CHAT_THREADS "
      f"({TOTAL_INPUT_TOKENS:,} tokens):\n")
print(projection.to_string(index=False))
print("\nSame data, ~13x cost swing between the smallest and frontier model - "
      "this is why model selection is the biggest cost lever.")

In [ ]:
# STEP 3 - Derive the credits-per-token rate from YOUR OWN actuals, then reconcile.
# CORTEX_FUNCTIONS_QUERY_USAGE_HISTORY records TOKENS and TOKEN_CREDITS per query, so
# TOKEN_CREDITS / TOKENS is the real blended rate you were charged - no rate table guessing.
actuals = session.sql("""
    SELECT
        model_name,
        SUM(tokens)                                            AS tokens,
        SUM(token_credits)                                     AS token_credits,
        SUM(token_credits) / NULLIF(SUM(tokens), 0) * 1e6      AS credits_per_1m_tokens
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_FUNCTIONS_QUERY_USAGE_HISTORY
    WHERE function_name ILIKE '%SENTIMENT%'
      AND tokens > 0
    GROUP BY model_name
    ORDER BY tokens DESC
""").collect()

if actuals:
    r = actuals[0]
    emp_rate = float(r['CREDITS_PER_1M_TOKENS'])
    emp_credits = TOTAL_INPUT_TOKENS / 1_000_000 * emp_rate
    print(f"Empirical rate from your history ({r['MODEL_NAME']}): "
          f"{emp_rate:.4f} credits / 1M tokens")
    print(f"-> full-run projection: {emp_credits:,.4f} credits "
          f"= ${emp_credits * PRICE_PER_CREDIT:,.2f} at ${PRICE_PER_CREDIT}/credit")
else:
    print("No SENTIMENT rows in CORTEX_FUNCTIONS_QUERY_USAGE_HISTORY yet.")
    print("ACCOUNT_USAGE views lag (can be a few hours), and a fresh account has no history.")
    print("Run the AI_SENTIMENT cells in Section 2 first, then re-run this cell later to")
    print("reconcile the rate-table projection above against what you were actually billed.")

In [ ]:
# STEP 4 - Reusable helper: price ANY function over ANY table/column before you run it.
# Uses the model-based COUNT_TOKENS primitive so it works for every model.
def estimate_cost(table, text_col, model,
                  credits_per_1m_tokens=None, price_per_credit=PRICE_PER_CREDIT,
                  where=None):
    """Project token/credit/$ cost of a row-wise AI function run for a given model.
    Pass credits_per_1m_tokens from the Service Consumption Table, or leave it None to
    auto-derive from CORTEX_FUNCTIONS_QUERY_USAGE_HISTORY for that model."""
    filt = f"WHERE {where}" if where else ""
    row = session.sql(f"""
        SELECT COUNT(*) AS n_rows,
               SUM(SNOWFLAKE.CORTEX.COUNT_TOKENS('{model}', {text_col})) AS tokens
        FROM {table} {filt}
    """).collect()[0]
    n_rows, tokens = int(row['N_ROWS']), int(row['TOKENS'] or 0)

    if credits_per_1m_tokens is None:
        hist = session.sql(f"""
            SELECT SUM(token_credits)/NULLIF(SUM(tokens),0)*1e6 AS rate
            FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_FUNCTIONS_QUERY_USAGE_HISTORY
            WHERE model_name = '{model}' AND tokens > 0
        """).collect()[0]['RATE']
        credits_per_1m_tokens = float(hist) if hist else None

    if credits_per_1m_tokens is None:
        print(f"{model} over {n_rows:,} rows: {tokens:,} input tokens "
              f"(no rate available - pass credits_per_1m_tokens to price it)")
        return
    credits = tokens / 1_000_000 * credits_per_1m_tokens
    print(f"{model} over {n_rows:,} rows of {table} (column '{text_col}'):")
    print(f"  {tokens:,} input tokens  ->  {credits:,.4f} credits  ->  "
          f"${credits * price_per_credit:,.2f}")
    print("  (input-text tokens only; task functions add a small prompt-template overhead)")
    return credits

# Example: price classifying every support ticket with the cheapest model first.
estimate_cost('SUPPORT_TICKETS', 'body', 'llama3.1-8b',
              credits_per_1m_tokens=CREDITS_PER_1M_TOKENS['llama3.1-8b'])

In [ ]:
# Daily credit consumption by function and model (last 30 days).
# Your single source of truth for AI Function spend.
session.sql("""
    SELECT
        DATE_TRUNC('day', start_time) AS usage_date,
        function_name,
        model_name,
        SUM(credits)                  AS total_credits,
        COUNT(DISTINCT query_id)      AS query_count
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY
    WHERE start_time >= DATEADD('day', -30, CURRENT_TIMESTAMP())
    GROUP BY 1, 2, 3
    ORDER BY usage_date DESC, total_credits DESC
""").show(30)

In [ ]:
# Top consumers: monthly credits by user. Join to USERS for email + default role
# so you know who to follow up with when spend spikes.
session.sql("""
    SELECT
        DATE_TRUNC('month', h.start_time) AS usage_month,
        u.name  AS user_name,
        u.email,
        u.default_role,
        SUM(h.credits)             AS total_credits,
        COUNT(DISTINCT h.query_id) AS query_count
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY h
    JOIN SNOWFLAKE.ACCOUNT_USAGE.USERS u ON h.user_id = u.user_id
    WHERE h.start_time >= DATEADD('month', -3, CURRENT_TIMESTAMP())
    GROUP BY 1, 2, 3, 4
    ORDER BY usage_month DESC, total_credits DESC
""").show(30)

### Cap spend with per-user quotas

A **per-user quota** is a first-class Snowflake object that enforces monthly (and optionally daily) per-user credit limits and can **auto-block** new AI requests once a user hits the limit — no custom tasks or scheduling. Blocks clear automatically at the cycle reset (UTC month / day) or when you raise the limit. Enforcement even terminates in-progress AI function calls.

Run the following as `ACCOUNTADMIN` (or a role granted `SNOWFLAKE.QUOTA_CREATOR` + `CREATE SNOWFLAKE.CORE.QUOTA` on a schema):

```sql
-- Create the quota, scope it to AI Functions, set the limit, turn on blocking
CREATE SNOWFLAKE.CORE.QUOTA my_quota();
CALL my_quota!ADD_SHARED_RESOURCE('AI FUNCTION');   -- whole AI Function domain
CALL my_quota!SET_PER_USER_LIMIT(120);              -- 120 credits / user / month
CALL my_quota!SET_PER_USER_LIMIT(20, 'DAILY');      -- optional daily cap
CALL my_quota!SET_BLOCK_ENFORCEMENT_ENABLED(TRUE);  -- auto-block at the limit

-- Inspect config and who is currently blocked
CALL my_quota!GET_CONFIG();
CALL my_quota!GET_ACTIVE_BLOCKS();
```

By default `SNOWFLAKE.CORTEX_USER` is granted to `PUBLIC`, so everyone can call AI Functions. To make limits un-bypassable, gate access behind a dedicated role:

```sql
USE ROLE ACCOUNTADMIN;
REVOKE DATABASE ROLE SNOWFLAKE.CORTEX_USER FROM ROLE PUBLIC;
-- then grant SNOWFLAKE.CORTEX_USER only through your AI_FUNCTIONS_USER_ROLE
```

> Quota block enforcement covers AI domains only (AI Functions, Cortex Agents, Snowflake CoWork, CoCo). Track warehouse spend in a separate quota. Prefer quotas over hand-rolled tasks; the [cost-management guide](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-func-cost-management) also shows an alert + task pattern if you need custom logic.

In [ ]:
# Find still-running AI Function queries and how much they've already cost.
# The usage view splits a long query into hourly windows; IS_COMPLETED is TRUE
# only on the final row, so a query is still running if NO row is completed yet.
running = session.sql("""
    WITH q AS (
        SELECT
            query_id,
            ANY_VALUE(user_id)           AS user_id,
            ARRAY_TO_STRING(ARRAY_AGG(DISTINCT function_name), ', ') AS functions,
            SUM(credits)                 AS credits_so_far,
            MIN(start_time)              AS started_at
        FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY
        WHERE start_time >= DATEADD('hour', -48, CURRENT_TIMESTAMP())
        GROUP BY query_id
        HAVING BOOLOR_AGG(is_completed) = FALSE   -- nothing completed => still running
    )
    SELECT q.query_id, u.name AS user_name, q.functions, q.credits_so_far, q.started_at
    FROM q
    LEFT JOIN SNOWFLAKE.ACCOUNT_USAGE.USERS u ON q.user_id = u.user_id
    ORDER BY q.credits_so_far DESC
""")
running.show()

# Kill a specific runaway query on the spot (needs OPERATE on its warehouse):
#   session.sql("SELECT SYSTEM$CANCEL_QUERY('<query_id>')").collect()
# Cancelling stops further cost but does NOT refund credits already consumed.

### Cost best practices

- **Prototype on a subset.** Develop against a `LIMIT` / sampled set; only run across the full table once the prompt and model are locked.
- **Pre-filter before the AI call.** Use `WHERE` (or a cheap `AI_FILTER`) to shrink the row set — every row is a model call.
- **Right-size the model.** Pick the smallest model that passes an AI Function Studio eval; larger models cost more per token.
- **Use aggregates at scale.** `AI_AGG` / `AI_SUMMARIZE_AGG` beat row-wise `AI_COMPLETE` for corpus-level questions and bypass context limits.
- **Don't oversize the warehouse.** AI Functions are token-bound, not compute-bound — a warehouse larger than `MEDIUM` doesn't speed them up, it just adds cost. Keep AI workloads on a small/medium warehouse.
- **Keep prompts tight.** Verbose instructions inflate input tokens on every row.
- **Tag for chargeback.** Set `QUERY_TAG` so `CORTEX_AI_FUNCTIONS_USAGE_HISTORY` can attribute spend by project or team.
- **Watch the latency.** The usage view lags up to ~5 minutes; combine it with quotas (near-real-time block) for hard caps.

---
## Summary

You built a customer-experience telemetry pipeline entirely in SQL:

- **AI_SENTIMENT** → overall and per-aspect sentiment
- **AI_CLASSIFY** → topic taxonomy
- **AI_EXTRACT** → structured fields from free text
- **AI_AGG / AI_SUMMARIZE_AGG** → theme discovery across the corpus
- **AI_FILTER** → churn-risk conversations, joined to customers
- **AI Function Studio** → create → evaluate → optimize a custom escalation router (LOW/MEDIUM/HIGH)
- **Cost & guardrails** → monitor spend in `CORTEX_AI_FUNCTIONS_USAGE_HISTORY`, cap it with per-user quotas, and kill runaway queries

The resulting `CX_TELEMETRY` table is consumed by the **Conversational BI** module, where a semantic view + Cortex Agent analyze it alongside churn and revenue. Swap the synthetic tables for your real chat logs and call transcripts — the SQL is unchanged.